# 第7章：建立聊天應用程式
## Azure OpenAI API 快速入門


## 概述
此筆記本改編自包含同時存取 [OpenAI](notebook-openai.ipynb) 服務的筆記本的 [Azure OpenAI 範例資源庫](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst)。

Python OpenAI API 也能與 Azure OpenAI 一起使用，僅需做一些修改。了解更多差異請參考此處：[如何使用 Python 在 OpenAI 與 Azure OpenAI 端點之間切換](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)

欲取得更多快速上手範例，請參考官方的 [Azure OpenAI 快速入門文件](https://learn.microsoft.com/azure/cognitive-services/openai/quickstart?pivots=programming-language-studio&WT.mc_id=academic-105485-koreyst)


## 目錄  

[概覽](#overview)  
[開始使用 Azure OpenAI 服務](#getting-started-with-azure-openai-service)  
[建立您的第一個提示詞](#build-your-first-prompt)  

[使用案例](#use-cases)    
[1. 摘要文字](#summarize-text)  
[2. 分類文字](#classify-text)  
[3. 產生新產品名稱](#generate-new-product-names)  
[4. 微調分類器](#fine-tune-a-classifier)  
[5. 內嵌向量](#embeddings)

[參考資料](#references)


### Azure OpenAI 服務快速入門

新客戶需要[申請存取權限](https://aka.ms/oai/access?WT.mc_id=academic-105485-koreyst)才能使用 Azure OpenAI 服務。  
核准完成後，客戶即可登入 Azure 入口網站，建立 Azure OpenAI 服務資源，並透過 Studio 開始試用模型。  

[快速入門的絕佳資源](https://techcommunity.microsoft.com/blog/educatordeveloperblog/azure-openai-service-is-now-generally-available/3719177?WT.mc_id=academic-105485-koreyst)


### 建立您的第一個提示  
這個簡短的練習將提供一個基本介紹，說明如何向 OpenAI 模型提交提示，以完成一個簡單的任務「摘要」。


<strong>步驟</strong>：  
1. 在您的 Python 環境中安裝 OpenAI 函式庫  
2. 載入標準輔助函式庫並設定您已建立的 OpenAI 服務的一般安全認證  
3. 為您的任務選擇一個模型  
4. 為模型建立一個簡單的提示  
5. 向模型 API 提交您的請求！


### 1. 安裝 OpenAI


  > [!NOTE] 如果在 Codespaces 上或在 Devcontainer 內運行此筆記本，此步驟並非必要


In [ ]:
%pip install openai python-dotenv

### 2. 匯入輔助程式庫並建立憑證實例


In [ ]:
import os
from openai import OpenAI
import numpy as np
from dotenv import load_dotenv
load_dotenv()

#validate data inside .env file

# The Responses API is served from the Azure OpenAI (Microsoft Foundry) v1 endpoint,
# so we point the OpenAI client at <your-endpoint>/openai/v1/ (no api_version needed).
endpoint = os.environ['AZURE_OPENAI_ENDPOINT']
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{endpoint.rstrip('/')}/openai/v1/",
  )


### 3. 找到合適的模型  
像 GPT-4o 和 GPT-4o mini 這樣的模型能夠理解並生成自然語言。Microsoft Foundry 中的 Azure OpenAI 提供一系列模型能力，每種能力具有不同的運算效能和速度，適合不同的任務需求。 

[Microsoft Foundry 中的 Azure OpenAI 模型](https://learn.microsoft.com/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst)



In [ ]:
# Select the deployment name configured in your .env file
model = os.environ['AZURE_OPENAI_DEPLOYMENT']


## 4. 提示設計  

「大型語言模型的魔力在於，透過訓練以在大量文字中最小化預測誤差，模型最終學會對這些預測有用的概念。例如，它們學會了這樣的概念」(1):

* 如何拼字
* 語法如何運作
* 如何改寫
* 如何回答問題
* 如何進行對話
* 如何以多種語言寫作
* 如何寫程式碼
* 等等。

#### 如何控制大型語言模型  
「在所有輸入到大型語言模型的資訊中，最具影響力的絕對是文字提示」(1)。

大型語言模型可以透過幾種方式被提示以產生輸出：

- 指令：告訴模型你想要什麼
- 補全：誘導模型補完你想要的開始部分
- 示範：向模型展示你想要什麼，可透過：
- 在提示中給幾個範例
- 在微調訓練資料集中給數百或數千個範例



#### 創建提示有三個基本準則：

<strong>展示並說明</strong>。透過指令、範例或兩者結合讓模型明確知道你想要什麼。如果你想讓模型依字母順序排列一個項目清單或按情緒分類一段文字，就要讓模型知道這正是你想要的。

<strong>提供高品質的資料</strong>。如果你想建立分類器或讓模型遵循某種模式，確保有足夠的範例。一定要校對你的範例 — 模型通常聰明到能看穿基本拼寫錯誤並給出回應，但它也可能認為這是故意的，進而影響回應。

**檢查設定。** temperature 與 top_p 設定控制模型生成回應時的確定性。如果你問的問題只有一個正確答案，就想把這些值設定較低；如果想要更豐富多樣的回應，則想設定得較高。人們在這些設定上最大錯誤是以為它們是「聰明度」或「創造力」的控制器。


資料來源：https://learn.microsoft.com/azure/ai-foundry/openai/overview


image 正在創建您的第一個文字提示！


### 5. 提交！


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### 重複相同的呼叫，結果會如何比較？ 


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## 摘要文本  
#### 挑戰  
透過在文本段落末加入「tl;dr:」來摘要文本。注意模型如何在沒有額外指令的情況下理解並執行多項任務。你可以用比 tl;dr 更具描述性的提示詞來實驗，以改變模型的行為並客製化所獲得的摘要(3)。  

近期的研究展示了透過在大型文本語料庫上進行預訓練，接著在特定任務上微調，在許多自然語言處理任務和基準測試中獲得了顯著提升。雖然這種方法在架構上通常與任務無關，但仍然需要數千至數萬個範例的任務專屬微調數據集。相比之下，人類通常只需少數範例或簡單指令就能執行新的語言任務——這是當前自然語言處理系統仍然大多難以做到的事。本文展示擴大語言模型規模能大幅提升與任務無關的少樣本表現，有時甚至能與先前最先進的微調方法競爭。  



摘要  


# 幾種使用情境練習  
1. 文字摘要  
2. 文字分類  
3. 產生新產品名稱
4. 內嵌表徵
5. 微調分類器


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\ntl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## 分類文本  
#### 挑戰  
將項目分類到推理時提供的類別。在以下範例中，我們在提示中同時提供類別和要分類的文本(*playground_reference)。 

客戶詢問：您好，我筆記型電腦鍵盤上的一個按鍵最近壞了，我需要更換： 

分類類別：


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## 產生新產品名稱
#### 挑戰
從示例詞中創造產品名稱。我們在提示中包含了將要為其產生名稱的產品資訊。我們也提供了類似的範例以顯示我們希望收到的模式。我們還將溫度值設定得較高，以增加隨機性和更具創新的回應。

產品描述：家用奶昔機
種子詞：快速、健康、緊湊。
產品名稱：HomeShaker、Fit Shaker、QuickShake、Shake Maker

產品描述：一雙能適合任何腳尺寸的鞋子。
種子詞：可調整、適合、全適合。


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


## 嵌入向量
本節將展示如何取得嵌入向量，並找出詞語、句子和文件之間的相似性。要執行以下筆記本，您需要部署一個以 `text-embedding-ada-002` 作為基礎模型的模型，並在 .env 檔案中使用 `AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT` 變數設定其部署名稱。


### 模型分類法 - 選擇嵌入模型

<strong>模型分類法</strong>：{family} - {capability} - {input-type} - {identifier}  

{family}     --> text-embedding  (嵌入模型家族)  
{capability} --> ada             (其他嵌入模型將於2024年退役)  
{input-type} --> n/a             (僅針對搜尋模型指定)  
{identifier} --> 002             (版本 002)  

model = 'text-embedding-ada-002'


  > [!NOTE] 如果您在 Codespaces 或 Devcontainer 內執行此筆記本，以下步驟非必要


In [ ]:
# Dependencies for embeddings_utils
%pip install matplotlib plotly scikit-learn pandas

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [ ]:
text = 'the quick brown fox jumped over the lazy dog'
model= os.environ['AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT']
client.embeddings.create(input='[text]', model=model).data[0].embedding

In [ ]:

# compare several words
automobile_embedding  = client.embeddings.create(input='automobile', model=model).data[0].embedding
vehicle_embedding     = client.embeddings.create(input='vehicle', model=model).data[0].embedding
dinosaur_embedding    = client.embeddings.create(input='dinosaur', model=model).data[0].embedding
stick_embedding       = client.embeddings.create(input='stick', model=model).data[0].embedding

print(cosine_similarity(automobile_embedding, vehicle_embedding))
print(cosine_similarity(automobile_embedding, dinosaur_embedding))
print(cosine_similarity(automobile_embedding, stick_embedding))

## 來自 CNN Daily News 資料集的文章比較
source: https://huggingface.co/datasets/cnn_dailymail


In [ ]:
import pandas as pd
cnn_daily_articles = ['BREMEN, Germany -- Carlos Alberto, who scored in FC Porto\'s Champions League final victory against Monaco in 2004, has joined Bundesliga club Werder Bremen for a club record fee of 7.8 million euros ($10.7 million). Carlos Alberto enjoyed success at FC Porto under Jose Mourinho. "I\'m here to win titles with Werder," the 22-year-old said after his first training session with his new club. "I like Bremen and would only have wanted to come here." Carlos Alberto started his career with Fluminense, and helped them to lift the Campeonato Carioca in 2002. In January 2004 he moved on to FC Porto, who were coached by José Mourinho, and the club won the Portuguese title as well as the Champions League. Early in 2005, he moved to Corinthians, where he impressed as they won the Brasileirão,but in 2006 Corinthians had a poor season and Carlos Alberto found himself at odds with manager, Emerson Leão. Their poor relationship came to a climax at a Copa Sul-Americana game against Club Atlético Lanús, and Carlos Alberto declared that he would not play for Corinthians again while Leão remained as manager. Since January this year he has been on loan with his first club Fluminense. Bundesliga champions VfB Stuttgart said on Sunday that they would sign a loan agreement with Real Zaragoza on Monday for Ewerthon, the third top Brazilian player to join the German league in three days. A VfB spokesman said Ewerthon, who played in the Bundesliga for Borussia Dortmund from 2001 to 2005, was expected to join the club for their pre-season training in Austria on Monday. On Friday, Ailton returned to Germany where he was the league\'s top scorer in 2004, signing a one-year deal with Duisburg on a transfer from Red Star Belgrade. E-mail to a friend .',
                        '(CNN) -- Football superstar, celebrity, fashion icon, multimillion-dollar heartthrob. Now, David Beckham is headed for the Hollywood Hills as he takes his game to U.S. Major League Soccer. CNN looks at how Bekham fulfilled his dream of playing for Manchester United, and his time playing for England. The world\'s famous footballer has begun a five-year contract with the Los Angeles Galaxy team, and on Friday Beckham will meet the press and reveal his new shirt number. This week, we take an in depth look at the life and times of Beckham, as CNN\'s very own "Becks," Becky Anderson, sets out to examine what makes the man tick -- as footballer, fashion icon and global phenomenon. It\'s a long way from the streets of east London to the Hollywood Hills and Becky charts Beckham\'s incredible rise to football stardom, a journey that has seen his skills grace the greatest stages in world soccer. She goes in pursuit of the current hottest property on the sports/celebrity circuit in the U.S. and along the way explores exactly what\'s behind the man with the golden boot. CNN will look back at the life of Beckham, the wonderfully talented youngster who fulfilled his dream of playing for Manchester United, his marriage to pop star Victoria, and the trials and tribulations of playing for England. We\'ll look at the highs (scoring against Greece), the lows (being sent off during the World Cup), the Man. U departure for the Galacticos of Madrid -- and now the Home Depot stadium in L.A. We\'ll ask how Beckham and his family will adapt to life in Los Angeles -- the people, the places to see and be seen and the celebrity endorsement. Beckham is no stranger to exposure. He has teamed with Reggie Bush in an Adidas commercial, is the face of Motorola, is the face on a PlayStation game and doesn\'t need fashion tips as he has his own international clothing line. But what does the star couple need to do to become an accepted part of Tinseltown\'s glitterati? The road to major league football in the U.S.A. is a well-worn route for some of the world\'s greatest players. We talk to some of the former greats who came before him and examine what impact these overseas stars had on U.S. soccer and look at what is different now. We also get a rare glimpse inside the David Beckham academy in L.A, find out what drives the kids and who are their heroes. The perception that in the U.S.A. soccer is a "game for girls" after the teenage years is changing. More and more young kids are choosing the European game over the traditional U.S. sports. E-mail to a friend .',
                        'LOS ANGELES, California (CNN) -- Youssif, the 5-year-old burned Iraqi boy, rounded the corner at Universal Studios when suddenly the little boy hero met his favorite superhero. Youssif has always been a huge Spider-Man fan. Meeting him was "my favorite thing," he said. Spider-Man was right smack dab in front of him, riding a four-wheeler amid a convoy of other superheroes. The legendary climber of buildings and fighter of evil dismounted, walked over to Youssif and introduced himself. Spidey then gave the boy from a far-away land a gentle hug, embracing him in his iconic blue and red tights. He showed Youssif a few tricks, like how to shoot a web from his wrist. Only this time, no web was spun. "All right Youssif!" Spider-Man said after the boy mimicked his wrist movement. Other superheroes crowded around to get a closer look. Even the Green Goblin stopped his villainous ways to tell the boy hi. Youssif remained unfazed. He didn\'t take a liking to Spider-Man\'s nemesis. Spidey was just too cool. "It was my favorite thing," the boy said later. "I want to see him again." He then felt compelled to add: "I know it\'s not the real Spider-Man." This was the day of dreams when the boy\'s nightmares were, at least temporarily, forgotten. He met SpongeBob, Lassie and a 3-year-old orangutan named Archie. The hairy, brownish-red primate took to the boy, grabbing his hand and holding it. Even when Youssif pulled away, Archie would inch his hand back toward the boy\'s and then snatch it. See Youssif enjoy being a boy again » . The boy giggled inside a play area where sponge-like balls shot out of toy guns. It was a far different artillery than what he was used to seeing in central Baghdad, as recently as a week ago. He squealed with delight and raced around the room collecting as many balls as he could. He rode a tram through the back stages at Universal Studios. At one point, the car shook. Fire and smoke filled the air, debris cascaded down and a big rig skidded toward the vehicle. The boy and his family survived the pretend earthquake unscathed. "Even I was scared," the dad said. "Well, I wasn\'t," Youssif replied. The father and mother grinned from ear to ear throughout the day. Youssif pushed his 14-month-old sister, Ayaa, in a stroller. "Did you even need to ask us if we were interested in coming here?" Youssif\'s father said in amazement. "Other than my wedding day, this is the happiest day of my life," he said. Just a day earlier, the mother and father talked about their journey out of Iraq and to the United States. They also discussed that day nine months ago when masked men grabbed their son outside the family home, doused him in gas and set him on fire. His mother heard her boy screaming from inside. The father sought help for his boy across Baghdad, but no one listened. He remembers his son\'s two months of hospitalization. The doctors didn\'t use anesthetics. He could hear his boy\'s piercing screams from the other side of the hospital. Watch Youssif meet his doctor and play with his little sister » . The father knew that speaking to CNN would put his family\'s lives in jeopardy. The possibility of being killed was better than seeing his son suffer, he said. "Anything for Youssif," he said. "We had to do it." They described a life of utter chaos in Baghdad. Neighbors had recently given birth to a baby girl. Shortly afterward, the father was kidnapped and killed. Then, there was the time when some girls wore tanktops and jeans. They were snatched off the street by gunmen. The stories can be even more gruesome. The couple said they had heard reports that a young girl was kidnapped and beheaded --and her killers sewed a dog\'s head on the corpse and delivered it to her family\'s doorstep. "These are just some of the stories," said Youssif\'s mother, Zainab. Under Saddam Hussein, there was more security and stability, they said. There was running water and electricity most of the time. But still life was tough under the dictator, like the time when Zainab\'s uncle disappeared and was never heard from again after he read a "religious book," she said. Sitting in the parking lot of a Target in suburban Los Angeles, Youssif\'s father watched as husbands and wives, boyfriends and girlfriends, parents and their children, came and went. Some held hands. Others smiled and laughed. "Iraq finished," he said in what few English words he knows. He elaborated in Arabic: His homeland won\'t be enjoying such freedoms anytime soon. It\'s just not possible. Too much violence. Too many killings. His two children have only seen war. But this week, the family has seen a much different side of America -- an outpouring of generosity and a peaceful nation at home. "It\'s been a dream," the father said. He used to do a lot of volunteer work back in Baghdad. "Maybe that\'s why I\'m being helped now," the father said. At Universal Studios, he looked out across the valley below. The sun glistened off treetops and buildings. It was a picturesque sight fit for a Hollywood movie. "Good America, good America," he said in English. E-mail to a friend . CNN\'s Arwa Damon contributed to this report.'
]

cnn_daily_article_highlights = ['Werder Bremen pay a club record $10.7 million for Carlos Alberto .\nThe Brazilian midfielder won the Champions League with FC Porto in 2004 .\nSince January he has been on loan with his first club, Fluminense .',
                                'Beckham has agreed to a five-year contract with Los Angeles Galaxy .\nNew contract took effect July 1, 2007 .\nFormer English captain to meet press, unveil new shirt number Friday .\nCNN to look at Beckham as footballer, fashion icon and global phenomenon .',
                                'Boy on meeting Spider-Man: "It was my favorite thing"\nYoussif also met SpongeBob, Lassie and an orangutan at Universal Studios .\nDad: "Other than my wedding day, this is the happiest day of my life"'
]

cnn_df = pd.DataFrame({"articles":cnn_daily_articles, "highligths":cnn_daily_article_highlights})

cnn_df.head()

In [ ]:
article1_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[0], model=model).data[0].embedding
article2_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[1], model=model).data[0].embedding
article3_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[2], model=model).data[0].embedding

print(cosine_similarity(article1_embedding, article2_embedding))
print(cosine_similarity(article1_embedding, article3_embedding))

# 參考資料  
- [Microsoft Foundry - Azure OpenAI Models](https://learn.microsoft.com/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  


# 需要更多協助  
[OpenAI 商業化團隊](AzureOpenAITeam@microsoft.com)


# 貢獻者
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
此文件已使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們努力追求準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於關鍵資訊，建議採用專業人工翻譯。我們不對因使用此翻譯所產生的任何誤解或誤譯承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
